# T09. Memory appears and disappears

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/cpython-internals/blob/main/lessons/t09-memory-appears-and-disappears/t09.ipynb)

T08 finished on the [reference count](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#reference-count): a small number in front of every object saying how many places are holding it. This lesson is about what happens when that number reaches zero, and about the one situation where it never does.

![the eight stages of the pipeline with none of them highlighted](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t09-memory-appears-and-disappears/diagrams/where-we-are.svg)

Nothing is highlighted again, for the same reason as last time. This is not a stage of the pipeline, it is what happens to the values underneath every stage, all the time, while the pipeline runs.

Most languages you have used have a garbage collector and you have never had to think about when it runs. Python is different in one important way: most of the freeing happens immediately, at a moment you can predict exactly, and only a small leftover case needs a collector at all. Knowing which is which is the difference between a `close()` you can rely on and one you cannot.

By the end you will be able to watch an object die, build one that refuses to, explain the trick the collector uses to tell garbage from live data, and say why freeing a big list does not make your process smaller.

## About the source references

Now and then this lesson points at CPython's own source, like this: `Include/refcount.h:417-429@v3.15.0rc1#Py_DECREF`.

Read it as four parts: the file, the lines, the release those line numbers belong to, and the name of the thing they are inside.

Every reference is a link, and every one is checked against the pinned source on each change, so a stale reference fails the build instead of sending you somewhere wrong. You never have to read any of it. The references are there so you can go deeper when you want to, and so you can check that this lesson is not making things up.

## Setup

Colab does not come with the small package these lessons use, so the next cell installs it. If you are running this from a checkout of the repository it is already installed and the cell does nothing.

In [ ]:
import sys

if sys.version_info < (3, 14):
    print("This lesson needs CPython 3.14 or newer.")
    print(f"This runtime is {sys.version.split()[0]}, and the cells below will not run on it.")
else:
    try:
        import pyxray
    except ImportError:
        %pip install -q "pyxray @ git+https://github.com/tamnd/cpython-internals@main#subdirectory=pyxray"
        import pyxray

## Which Python is this

Everything below was checked against the version this cell prints and against 3.14. Where the two disagree, the lesson says so.

In [ ]:
import pyxray

pyxray.show()

## Predict first

Two objects, each holding the other, and then both names go away.

```python
a = Node("a")
b = Node("b")
a.other = b
b.other = a
del a, b
```

Nothing in the program can reach either object any more. Are they gone?

Write your answer down, then run the cell.

In [ ]:
import gc
import weakref


class Node:
    def __init__(self, name):
        self.name = name
        self.other = None


gc.collect()

a = Node("a")
b = Node("b")
a.other = b
b.other = a

seen = weakref.ref(a)
del a, b

print("still there after both names went away ->", seen() is not None)
print("gc.collect() freed                     ->", gc.collect(), "objects")
print("still there now                        ->", seen() is not None)

True, then 2, then False.

The two objects outlived every name that could reach them, and stayed in memory until something else came along and swept them up. If you had written this in a loop you would have a program whose memory use climbs and never comes back down until the collector decides to run.

That is the shape of this lesson. Most of the time Python frees things the instant you let go of them, and this is the case where it does not, so the rest of the material is about why, and about what sits underneath.

## The count that decides everything

Every object carries a count of how many places are currently holding it. Assigning it to a name adds one, putting it in a list adds one, and the name going out of scope takes one away. When the count reaches zero the object is freed, right then, before the next line of your program runs.

![a table of six lines of Python and the reference count after each one](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t09-memory-appears-and-disappears/diagrams/the-count-moves.svg)

Here is the count moving. The demo is inside a function on purpose, because at the top level of a notebook the module's own dictionary holds an extra reference to everything and every number below would be one higher.

In [ ]:
from pyxray import obj


def watch():
    thing = [1, 2, 3]
    print(f"{'thing = [1, 2, 3]':<22} {obj.refcount(thing)}")

    holder = [thing]
    print(f"{'holder = [thing]':<22} {obj.refcount(thing)}")

    box = {"k": thing}
    print(f"{'box = {k: thing}':<22} {obj.refcount(thing)}")

    holder.clear()
    print(f"{'holder.clear()':<22} {obj.refcount(thing)}")

    del box
    print(f"{'del box':<22} {obj.refcount(thing)}")


watch()

1, 2, 3, 2, 1. When `watch` returns, `thing` goes out of scope, the count reaches zero, and the list is freed before the next statement in the notebook starts.

The C behind that is very short for a piece of code the whole language rests on.

[Include/refcount.h:417-429@v3.15.0rc1#Py_DECREF](https://github.com/python/cpython/blob/v3.15.0rc1/Include/refcount.h#L417-L429)

It subtracts one, and if the result is zero it frees the object. Everything else in this lesson is a consequence of those two lines, including the gap in them, which is that neither line ever fires for two objects that are only holding each other.

The immortality check at the top is the 3.14 change T08 ended on. `None`, `True`, small integers and every interned string skip the count entirely, so this function does nothing at all for a large fraction of the objects it is called on.

The work at zero happens in one function.

[Objects/object.c:3282-3300@v3.15.0rc1#_Py_Dealloc](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/object.c#L3282-L3300)

The comment above it is worth reading. Freeing a list frees everything in it, which can free everything in those, and a long enough chain of that is a stack overflow in the C code. CPython watches how much C stack is left and, when it gets close, puts the object on a queue to be freed later instead. That mechanism is called the trashcan, and it is the reason deleting a linked list of a million nodes does not crash the interpreter.

## Watching the moment it dies

You cannot observe a free with an ordinary variable, because holding the object is exactly what stops it happening. What you need is a reference that does not count, which is what a [weak reference](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#weak-reference) is for.

In [ ]:
from pyxray import heap


class Node:
    def __init__(self, name):
        self.name = name
        self.other = None


def plain():
    watcher = heap.Deaths()
    value = watcher.watch("value", Node("value"))
    print("after binding it  ", watcher.alive("value"))

    del value
    print("after del         ", watcher.alive("value"))
    print("freed so far      ", watcher.gone)


plain()

`heap.Deaths` is a thin wrapper over `weakref.ref` with a callback, and the callback is the interesting part. It runs at the moment the object is freed, which means it runs from inside the [deallocator](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#deallocation).

[Objects/weakrefobject.c:1001-1024@v3.15.0rc1#PyObject_ClearWeakRefs](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/weakrefobject.c#L1001-L1024)

Look at the third condition in that check: `Py_REFCNT(object) != 0`. This function refuses to run unless the count is already zero. By the time your callback fires, the object is past saving, which is why a weakref callback is handed the dead reference rather than the object.

Not everything can be watched this way. `int`, `str` and `tuple` have no room for a weak reference list, and neither does a class using `__slots__` without `__weakref__` in it. Trying gets you a clear error rather than a confusing one.

## When `__del__` runs

If a class defines `__del__`, that method is its [finalizer](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#finalizer), and it runs during the free. Because the free is immediate, so is `__del__`, and this is the property that makes people say Python has deterministic destruction.

In [ ]:
class Loud:
    def __init__(self, name):
        self.name = name
        self.other = None

    def __del__(self):
        print(f"  {self.name} is being freed")


def timing():
    print("before")
    x = Loud("x")
    print("bound")
    del x
    print("after")


timing()

The message lands between "bound" and "after", not at the end of the function and not at the end of the program.

It is worth being clear about how much you should lean on this. It is real, and it is genuinely useful, and it is also a CPython implementation detail. PyPy and other implementations do not reference count, so the same code there frees the object at some later point of the runtime's choosing. `with` blocks exist because they make the timing part of the language instead of part of the implementation. Use `__del__` for a last resort cleanup and `with` for cleanup you actually depend on.

## The shape counting cannot free

![two objects holding each other, before and after their names go away](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t09-memory-appears-and-disappears/diagrams/a-cycle.svg)

Both counts start at 2, one for the name and one for the other object. Dropping both names takes each count to 1 and stops there. Neither one ever reaches zero, so `Py_DECREF` never calls the deallocator, and the two objects sit in memory with nothing in the program able to reach them.

None of this is exotic. A doubly linked list is full of [reference cycles](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#reference-cycle), and so is a tree whose nodes carry a parent pointer. An exception traceback holds the [frame](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#frame) and the frame holds the exception. A [closure](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#closure) that refers to the function it lives in is a cycle. Any real program makes these constantly.

In [ ]:
import gc


def compare():
    watcher = heap.Deaths()

    lonely = watcher.watch("lonely", Node("lonely"))
    left = watcher.watch("left", Node("left"))
    right = watcher.watch("right", Node("right"))
    left.other = right
    right.other = left

    del lonely, left, right
    print("after dropping all three names:")
    print(watcher.report())

    print()
    print("collector freed", gc.collect(), "objects")
    print(watcher.report())


compare()

`lonely` is already gone before the collector is asked anything. The other two need it.

![reference counting and the cycle collector side by side](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t09-memory-appears-and-disappears/diagrams/two-ways-to-free.svg)

The important row is the last one on each side. Counting cannot free a cycle, and cycles are the only reason the [cycle collector](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#cycle-collector) exists. If you never made one, `gc.collect()` would have nothing to do.

## How the collector actually decides

The obvious way to write a garbage collector is to start from the things you know are alive, follow every reference, and free whatever you never reached. CPython does not do that, because it has no complete list of the roots. Instead it works out, for a group of candidate objects, whether the only things holding them are each other.

![the three steps the collector takes to tell garbage from live data](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t09-memory-appears-and-disappears/diagrams/the-subtract-trick.svg)

Step one takes a scratch copy of every candidate's real reference count, so the real counts are never touched.

[Python/gc.c:393-412@v3.15.0rc1#update_refs](https://github.com/python/cpython/blob/v3.15.0rc1/Python/gc.c#L393-L412)

Step two walks each candidate and subtracts one from the scratch copy of everything it points at. `tp_traverse` is the function every container type implements to say what it holds, and it is the same thing `gc.get_referents` gives you from Python.

[Python/gc.c:485-501@v3.15.0rc1#subtract_refs](https://github.com/python/cpython/blob/v3.15.0rc1/Python/gc.c#L485-L501)

The arithmetic is what pays off here. A scratch count of zero means every reference to that object came from inside the group. A count above zero means somebody outside is holding it, and that somebody was never on the candidate list, so it is alive and so is everything it can reach.

[Python/gc.c:566-583@v3.15.0rc1#move_unreachable](https://github.com/python/cpython/blob/v3.15.0rc1/Python/gc.c#L566-L583)

You can run the same idea from Python. `heap.cycles` walks the graph with `gc.get_referents` and finds the strongly connected components, which is the formal name for a group of objects that can all reach each other.

In [ ]:
def find():
    first = Node("first")
    second = Node("second")
    third = Node("third")
    first.other = second
    second.other = third
    third.other = first

    for cycle in heap.cycles(first):
        print(cycle.describe())


find()
gc.collect()

One cycle with three members, closed into a ring. The order the names print in is the order the search finished them in rather than the direction the references run, so read it as a membership list rather than a route.

`heap.cycles` hands back names rather than objects, which is a small decision worth explaining. Returning the objects would give you a fresh reference to each of them, and a tool for finding things that outlive their references should not be one of the reasons they are still here.

## `__del__` on a cycle

There used to be a nasty corner here. Before Python 3.4, a cycle whose members defined `__del__` could not be collected at all, because the collector had no safe order to run the finalizers in. Those objects went on a list called `gc.garbage` and stayed there for the life of the process.

PEP 442 fixed it. Finalizers now run before anything is freed, each one exactly once, and then the collection proceeds.

In [ ]:
def pep442():
    gc.collect()

    left = Loud("left")
    right = Loud("right")
    left.other = right
    right.other = left
    del left, right
    print("names gone, nothing has run yet")

    print("collector freed", gc.collect(), "objects")
    print("gc.garbage:", gc.garbage)


pep442()

Both `__del__` methods run, `gc.garbage` stays empty, and the memory comes back. If you find advice online about avoiding `__del__` because it leaks cycles, it was written before 2014.

[Python/gc.c:1041-1074@v3.15.0rc1#finalize_garbage](https://github.com/python/cpython/blob/v3.15.0rc1/Python/gc.c#L1041-L1074)

The `_PyGC_SET_FINALIZED` flag is what guarantees "exactly once". A finalizer is allowed to store `self` somewhere and bring the object back to life, and if that object dies again later the collector must not call the finalizer a second time.

## Generations

Running the whole subtract trick over every object in the process would be far too slow to do often. So the collector splits objects into three [generations](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#generation) by how long they have survived, and looks at the young group far more often than the old one.

![the three collector generations and how often each is examined](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t09-memory-appears-and-disappears/diagrams/generations.svg)

The bet is that most objects die young, which is true of nearly every Python program. The temporary list inside a loop is gone before the loop turns over. Anything still here after two collections is probably going to stay, so it gets looked at rarely.

In [ ]:
class Plain:
    def __init__(self):
        self.other = None


gc.collect()
value = Plain()

print("thresholds                 ", gc.get_threshold())
print("a fresh object is in       ", heap.generation_of(value))

gc.collect(0)
print("after one sweep of gen 0   ", heap.generation_of(value))

gc.collect(1)
print("after one sweep of gen 1   ", heap.generation_of(value))

print("what generation is 42 in?  ", heap.generation_of(42))

`(2000, 10, 10)` and then 0, 1, 2, and `None`.

Read the thresholds as three different kinds of number. The first one is a count of objects: when allocations minus frees since the last pass exceeds 2000, generation 0 is examined. The other two are counts of collections: after 10 passes over generation 0, generation 1 gets a look, and after 10 of those, generation 2 does.

[Include/internal/pycore_interp_structs.h:271-286@v3.15.0rc1#GC_GENERATION_INIT](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_interp_structs.h#L271-L286)

The first number was 700 for many years and became 2000 in 3.13. If you have read a blog post quoting 700, that is why.

The `None` at the end is the other half of the story. The collector does not track integers, and it does not track most strings either, because an object that cannot hold a reference to another object cannot possibly be part of a cycle. Not tracking them is the single largest thing the collector does for performance, since it removes most of the objects in a typical program from consideration entirely.

In [ ]:
gc.collect()

for value in [42, "text", (1, 2), [1, 2], {"k": 1}, (1, [2])]:
    print(f"{value!s:<12} tracked: {gc.is_tracked(value)}")

The two tuples are the interesting pair. A tuple is a container, so it starts out tracked, but a tuple holding only untracked things can never be on a cycle either. The collector notices this the first time it looks at one and stops tracking it. That is why `(1, 2)` prints False here and would print True if you built it and asked immediately, and why `(1, [2])` stays tracked forever.

## Where the bytes came from

Everything up to here has been about deciding when to free. Underneath that is a separate question: where the memory came from in the first place, and where it goes back to.

Python objects are small and there are a lot of them. A program that called the operating system's `malloc` for every 56 byte list would spend most of its time in the allocator. So CPython has its own allocator sitting on top, called [obmalloc](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#obmalloc), and it works in four layers.

![blocks inside a pool inside an arena inside the operating system](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t09-memory-appears-and-disappears/diagrams/the-allocator-layers.svg)

Your object sits in a [block](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#block). Blocks of the same size share a [pool](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#pool), pools share an [arena](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#arena), and only the arena ever talks to the operating system, which it does with one big request now and then rather than a small request constantly.

[Include/internal/pycore_obmalloc.h:216-226@v3.15.0rc1#ARENA_BITS](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_obmalloc.h#L216-L226)

[Include/internal/pycore_obmalloc.h:232-241@v3.15.0rc1#POOL_BITS](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_obmalloc.h#L232-L241)

There is a size limit on all of this. Ask for more than 512 bytes and the small object allocator steps aside and hands you to the system allocator instead.

[Include/internal/pycore_obmalloc.h:156-164@v3.15.0rc1#SMALL_REQUEST_THRESHOLD](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_obmalloc.h#L156-L164)

Under that limit, every request is rounded up to one of a fixed set of sizes.

![the size classes small requests are rounded up to](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t09-memory-appears-and-disappears/diagrams/size-classes.svg)

[Include/internal/pycore_obmalloc.h:128-146@v3.15.0rc1#ALIGNMENT](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_obmalloc.h#L128-L146)

Rounding up is what makes reuse cheap. A pool holds blocks of exactly one size, so a freed block fits any future object in the same class without any searching, measuring or splitting. The cost is a few wasted bytes per object and the benefit is an allocator fast enough that nobody thinks about it.

In [ ]:
ALIGNMENT = 16
THRESHOLD = 512

for want in [1, 16, 17, 56, 88, 500, 512, 513]:
    if want > THRESHOLD:
        served = "the system allocator"
    else:
        served = ALIGNMENT * ((want + ALIGNMENT - 1) // ALIGNMENT)
    print(f"ask for {want:>4} bytes -> {served}")

An empty list is 56 bytes and takes a 64 byte block, so 8 bytes go unused, and that is the price of the whole arrangement.

## Giving it back

Freeing a large object usually does not make your process smaller, which catches most people out the first time they measure it.

![the four steps between a count reaching zero and the operating system hearing about it](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t09-memory-appears-and-disappears/diagrams/giving-it-back.svg)

The block goes back to its pool.

[Objects/obmalloc.c:2594-2607@v3.15.0rc1#insert_to_freepool](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/obmalloc.c#L2594-L2607)

The pool stays in its arena. The arena goes back to the operating system only when every single pool inside it is empty, and even then only if it is not the last free arena, because a program that allocates and frees in a loop would otherwise thrash.

[Objects/obmalloc.c:681-700@v3.15.0rc1#_PyMem_ArenaFree](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/obmalloc.c#L681-L700)

One surviving object anywhere in an arena keeps the whole arena. This is why a process that peaked at two gigabytes usually still looks like it is using two gigabytes afterwards, and why the next allocation is fast. It is also why "Python has a memory leak" is usually "Python is holding onto arenas that are mostly empty".

You can watch the counts move with `sys.getallocatedblocks`, which counts blocks handed out rather than bytes.

In [ ]:
before = heap.allocated()
kept = [Plain() for _ in range(10_000)]
after = heap.allocated()

del kept
gc.collect()
freed = heap.allocated()

print("blocks in use at the start ", before)
print("after building 10000       ", after)
print("after dropping them        ", freed)

The middle number is about ten thousand higher and the last one is back where it started, so the blocks came back. Whether the operating system ever hears about it is a separate question, and the answer is usually no.

The clearest way to see that the memory really is reused is to watch an address come back.

In [ ]:
gc.collect()

first = Plain()
where = id(first)
del first

second = Plain()
print("first object was at ", hex(where))
print("second object is at ", hex(id(second)))
print("same address reused ->", id(second) == where)

The same address, almost every time. The first object freed its block back to a pool, and the very next request for that size class got the same block. This is also a good reminder that `id()` is only unique among objects that are alive at the same moment, which is the footnote T08 put on it.

## What to reach for

![a table of six questions and the tool that answers each one](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t09-memory-appears-and-disappears/diagrams/what-to-reach-for.svg)

Five of those six are in the standard library. The only thing this lesson needed its own code for is finding cycles, and that is because `gc` will tell you that it collected some objects but not which ones or what shape they were in.

## Try it yourself

**One.** Build a cycle with `gc.disable()` in force, in a loop, ten thousand times, and watch `sys.getallocatedblocks` climb. Then enable the collector and collect. Work out roughly how much memory each iteration was costing you.

**Two.** Write a class that resurrects itself in `__del__` by storing `self` in a module level list. Put two of them in a cycle, collect, and find out how many times each finalizer ran. Then explain the `_PyGC_SET_FINALIZED` flag from the source above without looking at it again.

**Three.** `gc.get_referrers(x)` tells you what is holding `x`. Use it on a list you have hidden inside a nested structure and find the container. Then work out why the answer includes a frame object and what that means for using this in a function.

**Four.** Find the size where an object stops coming from the pools and starts coming from the system allocator, using nothing but `sys.getallocatedblocks` and a loop over `bytes` objects of increasing length. The number will not be exactly 512 and working out the offset is the exercise.

**Five.** Take the tuple tracking result above and turn it into a rule. Build five containers, predict `gc.is_tracked` for each before and after a collection, and get all ten right.

## What just happened

Almost all freeing in CPython is immediate. The count drops to zero, the deallocator runs, and the memory is back before the next line of your program. That is why `__del__` fires when it does and why CPython feels different from a runtime with a tracing collector.

Counting has exactly one blind spot: a group of objects holding each other keeps every count above zero forever. The cycle collector exists for that case and no other.

It finds them by copying the counts, subtracting every reference that comes from inside the candidate group, and seeing what is left. Anything at zero was held only by its neighbours. Anything above zero has a holder outside the group and survives, along with everything it can reach.

It does this on a schedule based on three generations, because most objects die young and re-examining the survivors constantly would be wasted work. Objects that cannot hold references are not tracked at all, which removes most of a typical program from the problem.

Underneath both mechanisms is an allocator that hands out fixed size blocks from pools inside arenas. Freeing returns a block to a pool, and the arena goes back to the operating system only when it is completely empty, which is why your process rarely shrinks.

## Where this goes next

You now know what an object is and what happens to it from the moment it is built to the moment its memory is reused. T10 puts the whole first part together on one page: the pipeline from T02 through T07, the object model from T08, and this, drawn as one diagram you can keep next to you for the rest of the material.